# Task 4.1b. Tinh chỉnh GCN trên Graph A

| | |
|---|---|
| **Mục đích** | So sánh từng yếu tố với GCN-A mặc định, chọn bằng validation |
| **Đầu vào** | Kaggle Dataset `DiverseVul 9077 - Processed Graph Data` |
| **Đầu ra** | `results_4_1b_gcn_tuning/<ID>/`: `results.csv`, `per_class.csv`, `confusion_matrices.json`, `training_history.json`, `summary.md`, metadata và trọng số; lựa chọn đã cố định và kết quả test riêng |
| **Máy chạy** | Kaggle GPU (T4 hoặc P100), bật Internet |

Hai protocol `project_wise`, `seen_project`; seed 42, 43, 44. Dùng chung model, trainer, metric. Không sửa dữ liệu, vocabulary, nhãn hoặc split. Số cấu hình phải được nhóm thống nhất trước khi chạy. Test chỉ dùng sau khi cố định cấu hình.


In [1]:
from pathlib import Path
import csv
import json
import os
import re
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/meo225/software-bug-detection-using-graphs.git"
APPROVED_BRANCH = "feature/4_1b_gcn_tuning"
APPROVED_COMMIT = "5d8b99e821c0eb93d873f6ff17c3c7fe8ab5d5f5"  # SHA đầy đủ đã được phê duyệt, chứa triển khai
DATASET_ID = "vtnmai/diversevul-9077-processed-graph-data/versions/1"  # vtnmai/diversevul-9077-processed-graph-data/versions/<version>
APPROVED_CONFIG_IDS = ["B0", "E1", "D1", "W1", "H1", "L1", "N1"]  # Danh sách đã thống nhất: B0, E1, D1, W1, H1, L1, N1
RUN_TUNING = True
RUN_FINAL_TEST = False
# Để phục hồi hoặc chỉ đánh giá test: trỏ tới output đã lưu, có frozen_selection.json.
SAVED_RESULTS_ROOT = None  # ví dụ /kaggle/input/<saved-output>/results_4_1b_gcn_tuning
OUT_DIR = Path("/kaggle/working/results_4_1b_gcn_tuning")
FINAL_OUT_DIR = Path("/kaggle/working/results_4_1b_gcn_final")
ACTIVE = RUN_TUNING or RUN_FINAL_TEST
if ACTIVE:
    if not Path("/kaggle/input").is_dir():
        raise RuntimeError("Chỉ chạy trên Kaggle với dataset chính thức.")
    if not re.fullmatch(r"[0-9a-f]{40}", APPROVED_COMMIT):
        raise ValueError("Cần SHA đầy đủ đã được phê duyệt.")
    if not re.fullmatch(r"vtnmai/diversevul-9077-processed-graph-data/versions/[1-9][0-9]*", DATASET_ID):
        raise ValueError("Ghi phiên bản Kaggle dataset đã gắn vào notebook.")
REPO_DIR = Path("/tmp/g41b") / APPROVED_COMMIT
READ_ROOT = Path(SAVED_RESULTS_ROOT) if SAVED_RESULTS_ROOT else OUT_DIR


## 1. Tải mã nguồn và ghi phiên bản

Nhánh phải có commit đã duyệt trên repository chính thức. Không yêu cầu đã merge vào `main`. Không dùng `git pull`, không cài thư viện trong notebook. Giữ output notebook để lưu nhật ký chạy.


In [2]:
if ACTIVE:
    if not REPO_DIR.exists():
        REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", "--no-checkout", "--single-branch", "--branch",
                        APPROVED_BRANCH, REPO_URL, str(REPO_DIR)], check=True, timeout=180)
    subprocess.run(["git", "checkout", "--detach", APPROVED_COMMIT], cwd=REPO_DIR, check=True)
    actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
    assert actual == APPROVED_COMMIT
    sys.path.insert(0, str(REPO_DIR))
    import torch
    from src.training.gcn_tuning import (
        BASELINE, CANDIDATES, HYPOTHESES, PROTOCOLS, SEEDS,
        configuration, code_identity, environment, freeze_winner, write_json,
        reuse_candidate, read_frozen, validate_context, validate_final,
    )
    from src.graph.processed import ids_hash, load_schema
    assert torch.cuda.is_available(), "Bật GPU trong Kaggle Settings."
    assert torch.ones(1, device="cuda").sum().item() == 1, "Không khởi tạo được GPU."
    assert not code_identity(REPO_DIR)["dirty"], "Checkout phải sạch."
    print("Revision:", actual, "Environment:", environment(), flush=True)


def run_script(arguments, log_path):
    cmd = [sys.executable, "scripts/train_gnn.py", *arguments]
    print("Command:", cmd, flush=True)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("x", encoding="utf-8") as handle:
        proc = subprocess.Popen(cmd, cwd=REPO_DIR, text=True, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT,
                                env={**os.environ, "PYTHONUNBUFFERED": "1"})
        for line in proc.stdout:
            print(line, end="", flush=True)
            handle.write(line)
            handle.flush()
        if proc.wait():
            raise RuntimeError(f"Script thất bại; giữ output và xem {log_path}")


Cloning into '/tmp/g41b/5d8b99e821c0eb93d873f6ff17c3c7fe8ab5d5f5'...
HEAD is now at 5d8b99e feat(gnn): implement Task 4.1b GCN tuning


Revision: 5d8b99e821c0eb93d873f6ff17c3c7fe8ab5d5f5 Environment: {'python': '3.13.16', 'torch': '2.11.0+cu130', 'numpy': '2.1.3', 'sklearn': '1.6.1', 'cuda': '13.0', 'gpu': 'Tesla T4'}


## 2. Dữ liệu và cấu hình

Mỗi biến thể chỉ đổi một yếu tố so với B0: E1 dừng theo validation loss; D1 dropout 0,4; W1 weight decay 1e-3; H1 dim 64; L1 hai lớp GCN; N1 tối đa 1.000 node. E1 vẫn giữ mô hình có Macro-F1 validation chưa làm mượt cao nhất. Các cấu hình được so bằng trung bình Macro-F1 validation `project_wise`; `seen_project` là đối chứng.

B0 của 4.1a là tham chiếu lịch sử. Chỉ tái sử dụng làm đối chứng trực tiếp sau khi xác minh tính tương đương của dữ liệu, mã và môi trường. Các CSV cũ không có trọng số hoặc đầy đủ provenance để dùng trực tiếp cho bước đánh giá test của notebook này. Mặc định đề xuất chạy B0 trong cùng môi trường; đây không phải ngân sách đã duyệt.


In [3]:
if ACTIVE:
    found = [p.parent for p in Path("/kaggle/input").rglob("metadata.json")
             if all((p.parent / protocol / "vocab.json").is_file() for protocol in PROTOCOLS)]
    assert len(found) == 1, "Gắn đúng một phiên bản dataset processed chính thức."
    PROCESSED = found[0]
    metadata = json.loads((PROCESSED / "metadata.json").read_text(encoding="utf-8"))
    assert metadata.get("spec_version") in {"1.1", "1.2"}
    assert metadata.get("n_valid_samples") == 7799
    assert sorted(load_schema(REPO_DIR / "configs/graph/schema_spec.yaml")["cwe_mapping"].values()) == list(range(23))
    from collections import Counter
    expected = {"project_wise": {"train": 6101, "validation": 762, "test": 936},
                "seen_project": {"train": 6272, "validation": 764, "test": 763}}
    for protocol in PROTOCOLS:
        with (PROCESSED / protocol / "manifest.csv").open(encoding="utf-8", newline="") as handle:
            manifest = list(csv.DictReader(handle))
        counts = Counter(row["split"] for row in manifest)
        assert len({row["sample_id"] for row in manifest}) == len(manifest)
        assert all(0 <= int(row["y"]) < 23 for row in manifest)
        vocab = json.loads((PROCESSED / protocol / "vocab.json").read_text(encoding="utf-8"))
        assert isinstance(vocab["k"], int) and vocab["k"] > 0
        for name in ("node_type", "token"):
            assert sorted(vocab[name].values()) == list(range(len(vocab[name])))
            assert vocab[name].get("<PAD>") == 0 and vocab[name].get("<UNK>") == 1
        assert vocab["token"].get("<MASK>") == 2
        assert vocab.get("train_ids_sha256") == ids_hash(row["sample_id"] for row in manifest if row["split"] == "train")
        assert dict(counts) == expected[protocol]
        for split in expected[protocol]:
            assert (PROCESSED / protocol / f"graph_a_{split}.pt").is_file()
    print("Dataset:", DATASET_ID, PROCESSED)
if RUN_TUNING:
    assert APPROVED_CONFIG_IDS and len(set(APPROVED_CONFIG_IDS)) == len(APPROVED_CONFIG_IDS)
    assert all(key in CANDIDATES for key in APPROVED_CONFIG_IDS)
    for key in APPROVED_CONFIG_IDS:
        cfg = configuration(key)
        changed = {k: v for k, v in cfg.items() if v != BASELINE[k]}
        assert len(changed) == (0 if key == "B0" else 1)
        print(key, changed, HYPOTHESES[key], flush=True)
    print("Số cấu hình:", len(APPROVED_CONFIG_IDS), "Số lần train tối đa:", 6 * len(APPROVED_CONFIG_IDS))


Dataset: vtnmai/diversevul-9077-processed-graph-data/versions/1 /kaggle/input/datasets/vtnmai/diversevul-9077-processed-graph-data/processed
B0 {} Reproduce the official default in the tuning environment.
E1 {'early_stopping': 'val_loss'} Validation-loss stopping may be less sensitive to noisy Macro-F1 fluctuations.
D1 {'dropout': 0.4} More dropout may reduce early overfitting.
W1 {'weight_decay': 0.001} Stronger weight decay may improve generalization.
H1 {'dim': 64} Smaller hidden dimension may reduce memorization.
L1 {'num_layers': 2} Two GCN layers may reduce excessive propagation and model capacity.
N1 {'max_nodes': 1000} A 1000-node cap tests sensitivity to large graphs and aggregation cost.
Số cấu hình: 7 Số lần train tối đa: 42


## 3. Train chỉ validation

`RUN_TUNING=False` bỏ qua bước này. Khi bật, chạy cấu hình chưa có output; cấu hình đã hoàn tất chỉ được dùng lại sau khi kiểm tra đủ artifact, metadata, code, dataset và môi trường. Có thể đặt `SAVED_RESULTS_ROOT` tới output của phiên trước: cấu hình hợp lệ được chép sang `OUT_DIR` mới, không train lại. Output thiếu hoặc không nhất quán gây lỗi rõ ràng; giữ nguyên để kiểm tra, chọn thư mục output mới khi cần chạy lại cấu hình dở dang. Không tự tiếp tục optimizer của lần train bị ngắt. Không nạp test ở bước này.


In [4]:
if RUN_TUNING:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    plan = {
        "code": code_identity(REPO_DIR), "dataset_id": DATASET_ID,
        "configurations": {key: configuration(key) for key in APPROVED_CONFIG_IDS},
        "hypotheses": {key: HYPOTHESES[key] for key in APPROVED_CONFIG_IDS},
        "protocols": list(PROTOCOLS), "seeds": list(SEEDS),
        "selection": "project_wise.validation.macro_f1.mean",
        "checkpoint_selection": "maximum raw validation Macro-F1; earliest tie",
    }
    plan_path = OUT_DIR / "experiment_plan.json"
    if plan_path.exists():
        if json.loads(plan_path.read_text(encoding="utf-8")) != plan:
            raise ValueError("Kế hoạch đã lưu khác cấu hình hiện tại; không ghi đè.")
    else:
        with plan_path.open("x", encoding="utf-8") as handle:
            json.dump(plan, handle, ensure_ascii=False, indent=2)
    for key in APPROVED_CONFIG_IDS:
        target = OUT_DIR / key
        source = target if target.exists() else READ_ROOT / key
        if source.exists():
            reuse_candidate(source, key, REPO_DIR, PROCESSED,
                            REPO_DIR / "configs/graph/schema_spec.yaml", DATASET_ID)
            if source.resolve() != target.resolve():
                shutil.copytree(source, target)  # destination must be unused
        else:
            args = ["--processed-dir", str(PROCESSED), "--out-dir", str(target),
                    "--models", "gcn", "--graphs", "graph_a", "--protocols", *PROTOCOLS,
                    "--seeds", *map(str, SEEDS), "--validation-only", "--save-checkpoints",
                    "--config-id", key, "--dataset-id", DATASET_ID]
            for name, value in configuration(key).items():
                if value is not None:
                    args += ["--" + name.replace("_", "-"), str(value)]
            run_script(args, OUT_DIR / f"{key}.log")
            reuse_candidate(target, key, REPO_DIR, PROCESSED,
                            REPO_DIR / "configs/graph/schema_spec.yaml", DATASET_ID)


Command: ['/usr/bin/python3', 'scripts/train_gnn.py', '--processed-dir', '/kaggle/input/datasets/vtnmai/diversevul-9077-processed-graph-data/processed', '--out-dir', '/kaggle/working/results_4_1b_gcn_tuning/B0', '--models', 'gcn', '--graphs', 'graph_a', '--protocols', 'project_wise', 'seen_project', '--seeds', '42', '43', '44', '--validation-only', '--save-checkpoints', '--config-id', 'B0', '--dataset-id', 'vtnmai/diversevul-9077-processed-graph-data/versions/1', '--dim', '128', '--num-layers', '3', '--dropout', '0.2', '--lr', '0.001', '--weight-decay', '0.0001', '--batch-size', '32', '--max-epochs', '40', '--patience', '8', '--early-stopping', 'macro_f1']
Thiết bị: cuda; torch 2.11.0+cu130
project_wise/graph_a: train=6101, validation=762; K=23
  [gcn seed=42] epoch 01 loss=3.1383 val_macro_f1=0.0166
  [gcn seed=42] epoch 02 loss=3.0719 val_macro_f1=0.0068
  [gcn seed=42] epoch 03 loss=2.8364 val_macro_f1=0.0251
  [gcn seed=42] epoch 04 loss=2.6250 val_macro_f1=0.0324
  [gcn seed=42] e

## 4. Cố định lựa chọn bằng validation

Xếp hạng bằng trung bình Macro-F1 validation chưa làm tròn của `project_wise` trên ba seed. Khi bằng nhau ưu tiên B0, E1, D1, W1, H1, L1, N1. Dùng cùng cấu hình cho cả hai protocol. Nếu đã có lựa chọn, chỉ nạp và xác minh; không thay bằng cấu hình khác. Không chọn theo test. Nếu không chạy tuning, bước này chỉ nạp lựa chọn khi `RUN_FINAL_TEST=True`.


In [5]:
if RUN_TUNING or RUN_FINAL_TEST:
    schema = REPO_DIR / "configs/graph/schema_spec.yaml"
    FROZEN_ROOT = OUT_DIR if RUN_TUNING else READ_ROOT
    FROZEN = FROZEN_ROOT / "frozen_selection.json"
    saved_frozen = READ_ROOT / "frozen_selection.json"
    if RUN_TUNING and not FROZEN.exists() and saved_frozen.exists():
        previous = read_frozen(saved_frozen, REPO_DIR, PROCESSED, schema, READ_ROOT)
        if {r["config_id"] for r in previous["ranking"]} != set(APPROVED_CONFIG_IDS):
            raise ValueError("Danh sách cấu hình khác lựa chọn đã cố định; không tự chọn lại.")
        with FROZEN.open("xb") as handle:
            handle.write(saved_frozen.read_bytes())
    if FROZEN.exists():
        frozen = read_frozen(FROZEN, REPO_DIR, PROCESSED, schema, FROZEN_ROOT)
        if RUN_TUNING and {r["config_id"] for r in frozen["ranking"]} != set(APPROVED_CONFIG_IDS):
            raise ValueError("Danh sách cấu hình khác lựa chọn đã cố định; không ghi đè.")
    elif RUN_TUNING:
        frozen = freeze_winner([OUT_DIR / key for key in APPROVED_CONFIG_IDS], FROZEN)
    else:
        raise FileNotFoundError(f"Thiếu lựa chọn validation đã cố định: {FROZEN}")
    validate_context(frozen["winner"]["metadata"], REPO_DIR, PROCESSED, schema, DATASET_ID)
    print("Validation ranking:", frozen["ranking"], flush=True)
    print("Frozen configuration:", frozen["winner"]["config_id"], flush=True)


Validation ranking: [{'config_id': 'L1', 'score': 0.06958665231149123}, {'config_id': 'H1', 'score': 0.06321407529734191}, {'config_id': 'N1', 'score': 0.0594369634345724}, {'config_id': 'W1', 'score': 0.05590380480392897}, {'config_id': 'B0', 'score': 0.05452052013497637}, {'config_id': 'D1', 'score': 0.048967718905094514}, {'config_id': 'E1', 'score': 0.043418551187484986}]
Frozen configuration: L1


## 5. Đánh giá test của cấu hình đã cố định

`RUN_FINAL_TEST=False` bỏ qua test. Trong phiên riêng: gắn dataset processed và output tuning đã lưu, đặt `SAVED_RESULTS_ROOT`, cùng commit và dataset version, giữ `RUN_TUNING=False`, bật `RUN_FINAL_TEST=True`. Script xác minh lựa chọn và nạp trọng số theo results root mới, không train lại. `FINAL_OUT_DIR` phải là thư mục ghi được, riêng với output tuning. Nếu đã có kết quả final hợp lệ thì xác minh và dùng lại; không ghi đè. Kết quả final dở dang gây lỗi và cần một thư mục final mới. Báo cạnh 4.1a và token baseline, nêu rõ seen_project có 5 validation và 6 test trùng input với train.


In [6]:
if RUN_FINAL_TEST:
    if FINAL_OUT_DIR.exists() and any(FINAL_OUT_DIR.iterdir()):
        validate_final(FINAL_OUT_DIR, FROZEN, REPO_DIR, PROCESSED, schema, FROZEN_ROOT)
    else:
        run_script(["--processed-dir", str(PROCESSED), "--out-dir", str(FINAL_OUT_DIR),
                    "--frozen-selection", str(FROZEN), "--results-root", str(FROZEN_ROOT)],
                   FINAL_OUT_DIR.parent / (FINAL_OUT_DIR.name + ".log"))
        validate_final(FINAL_OUT_DIR, FROZEN, REPO_DIR, PROCESSED, schema, FROZEN_ROOT)
    print((FINAL_OUT_DIR / "summary.md").read_text(encoding="utf-8"))
